# KYC Intelligence — End-to-End Walkthrough

This notebook walks through the entire stack: GraphDB ontology + GLEIF data + Neo4j analytics + GraphRAG agent.

**Prerequisites** (run once from a terminal):
```bash
docker compose up -d
pip install -r requirements.txt
```

Then go module-by-module below.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.kg_client import GraphDBClient, Neo4jClient, graphdb_healthy, neo4j_healthy

print(f"GraphDB healthy: {graphdb_healthy()}")
print(f"Neo4j healthy:   {neo4j_healthy()}")

## Module 1 — GraphDB foundation

Run scripts 01-03 from a terminal first, then verify here:
```bash
python scripts/01_setup_graphdb.py
python scripts/02_load_fibo.py
python scripts/03_load_fibo2glei_mapping.py
```

In [ ]:
gdb = GraphDBClient()
print(f"Repository: {gdb.repo}")
print(f"Total triples: {gdb.count_triples():,}\n")
print("Named graphs:")
for graph, n in gdb.list_named_graphs():
    print(f"  {n:>10,}  {graph}")

### SPARQL: subclass hierarchy of FIBO LegalPerson

In [ ]:
rows = gdb.query("""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX fibo-be: <https://spec.edmcouncil.org/fibo/ontology/BE/LegalEntities/LegalPersons/>
    SELECT DISTINCT ?descendant ?label WHERE {
        ?descendant rdfs:subClassOf* fibo-be:LegalPerson .
        OPTIONAL { ?descendant rdfs:label ?label }
    } ORDER BY ?label LIMIT 20
""")
for r in rows: print(f"  {r.get('label', r['descendant'])}")

### SPARQL: confirm KYC↔FIBO mapping is in place

In [ ]:
gdb.ask("""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX kyc:     <http://kyc-kg.example.org/ontology#>
    PREFIX fibo-be: <https://spec.edmcouncil.org/fibo/ontology/BE/LegalEntities/LegalPersons/>
    ASK { kyc:LegalEntity rdfs:subClassOf+ fibo-be:LegalPerson }
""")

## Module 2 — Synthetic data + Neo4j load
```bash
python scripts/06_generate_synthetic_data.py
python scripts/07_load_neo4j.py
```

In [ ]:
neo = Neo4jClient()
print(f"Total nodes:           {neo.node_count():,}")
print(f"  LegalEntity:        {neo.node_count('LegalEntity'):,}")
print(f"  NaturalPerson:      {neo.node_count('NaturalPerson'):,}")
print(f"  SanctionedEntity:   {neo.node_count('SanctionedEntity'):,}")
print(f"  PoliticallyExposed: {neo.node_count('PoliticallyExposedPerson'):,}")
print(f"\nLabels: {neo.list_labels()}")

## Module 3 — Cypher: UBO discovery
Variable-length traversal up the ownership chain to a controlling natural person.

In [ ]:
import pandas as pd
rows = neo.query("""
    MATCH path = (e:LegalEntity)-[:DIRECTLY_OWNED_BY*0..6]->()-[:CONTROLLED_BY]->(p:NaturalPerson)
    WHERE p.isSanctioned = true
    RETURN e.id AS entity, e.name AS entity_name,
           p.name AS sanctioned_ubo, length(path) AS hops
    ORDER BY hops DESC
    LIMIT 10
""")
pd.DataFrame(rows)

## Module 4 — GDS analysis
```bash
python scripts/08_gds_analysis.py
```
Then visualise the resulting risk scores:

In [ ]:
rows = neo.query("""
    MATCH (e:LegalEntity) WHERE e.kycRiskScore IS NOT NULL
    RETURN e.kycRiskScore AS score, count(*) AS n
    ORDER BY score
""")
df = pd.DataFrame(rows)
df.plot.bar(x='score', y='n', title='Distribution of KYC risk scores', figsize=(10,4))

### Top 10 risky entities

In [ ]:
pd.DataFrame(neo.query("""
    MATCH (e:LegalEntity) WHERE e.kycRiskScore > 0
    RETURN e.id AS id, e.name AS name, e.jurisdiction AS jur,
           e.kycRiskScore AS score
    ORDER BY score DESC LIMIT 10
"""))

## Module 5 — Circular ownership (SCC)

In [ ]:
rings = neo.query("""
    MATCH (e:LegalEntity) WHERE e.sccComponentId IS NOT NULL
    WITH e.sccComponentId AS scc, collect(e.id) AS members
    WHERE size(members) > 1
    RETURN scc, members
""")
for r in rings: print(f"Ring #{r['scc']}: {' → '.join(r['members'])} → ...")

## Module 6 — SHACL validation
```bash
python scripts/10_shacl_validate.py
```

## Module 7 — GraphRAG agent
Requires `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` in `.env`.

In [ ]:
from dotenv import load_dotenv; load_dotenv()
if not (os.getenv('ANTHROPIC_API_KEY') or os.getenv('OPENAI_API_KEY')):
    print('⚠️  No LLM key set — skipping agent demo. Add one to .env to try.')
else:
    import importlib.util
    spec = importlib.util.spec_from_file_location('agent_mod', '../scripts/09_graphrag_agent.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    agent = mod.build_agent()
    cfg = {'configurable': {'thread_id': 'notebook-demo'}}

    q = 'Show me the top 5 riskiest entities and explain why.'
    out = agent.invoke({'messages': [('user', q)]}, config=cfg)
    print(out['messages'][-1].content)

---

✅ End of walkthrough. Next: explore [dashboard/app.py](../dashboard/app.py) for the live UI, or run `pytest -m integration` to verify all planted patterns are detected.